# VLA는 지시를 정말 이해할까? — SmolVLA × LIBERO 언어 지시 테스트

**실험 아이디어**: 로봇이 보는 장면과 성공 기준은 그대로 두고, **언어 지시문만** 바꿔서 성공률이 어떻게 변하는지 봅니다.

| 조건 | 정책에 들어가는 지시문 (예: 0번 과제) | 언어를 이해한다면 기대되는 결과 |
|---|---|---|
| `original` | pick up the alphabet soup and place it in the basket | 높은 성공률 (기준선) |
| `paraphrase` | grab the alphabet soup and put it into the basket | 기준선과 비슷해야 함 |
| `wrong_object` | pick up the **salad dressing** and place it in the basket (같은 장면에 있는 다른 물체) | 원래 과제 성공률이 **떨어져야** 함 |
| `nonsense` | what is the weather like today | 원래 과제 성공률이 **떨어져야** 함 |

모든 조건에서 **원래 과제(알파벳 수프를 바구니에)의 성공 여부**를 잽니다. `wrong_object`나 `nonsense`에서도 성공률이 높게 나오면, 정책이 문장보다 **화면 배치를 보고 과제를 추측**한다는 증거입니다.

- 모델: [HuggingFaceVLA/smolvla_libero](https://huggingface.co/HuggingFaceVLA/smolvla_libero) (SmolVLA 4.5억 파라미터, LIBERO로 학습)
- 시뮬레이터: LIBERO-Object (MuJoCo, 10개 과제)
- 도구: LeRobot 0.6.1

## 시작 전에
1. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 선택
2. 위에서부터 셀을 **순서대로** 실행 (Shift+Enter)
3. 3번 설정 셀에서 **Google Drive 연결을 허용**하세요. 결과는 Drive(`내 드라이브/vla_results_full`)에 저장되어 런타임이 끊겨도 남습니다.
4. 런타임이 끊기면 설치(`/content`)는 지워지므로 1번 셀부터 다시 실행하세요. 이미 끝난 조건은 Drive에 결과가 있으면 건너뜁니다.

## 0. GPU 확인
`Tesla T4` 같은 GPU 이름이 보여야 합니다. `command not found`가 나오면 런타임 유형을 GPU로 바꾸세요.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. 설치 (5~10분)
Colab 기본 파이썬과 섞이지 않도록 **Python 3.12 가상환경**(`/content/venv`)을 따로 만들고 LeRobot과 LIBERO를 설치합니다.
LIBERO는 처음 불러올 때 설정 경로를 묻는데, 자동으로 `n`(기본 경로 사용)을 답하게 했습니다.

In [ ]:
%%bash
set -e
pip install -q uv
uv venv /content/venv --python 3.12 -q
uv pip install --python /content/venv/bin/python -q "lerobot[libero,smolvla]==0.6.1"
echo n | /content/venv/bin/python -c "import libero.libero" > /dev/null
/content/venv/bin/python - <<'EOF'
import importlib.metadata as md, torch
print("lerobot", md.version("lerobot"), "| torch", torch.__version__, "| CUDA 사용 가능:", torch.cuda.is_available())
EOF

## 2. 실험 스크립트 작성
`lerobot-eval`을 감싸서 **지시문만 바꿔치기**하는 스크립트입니다. LeRobot은 매 스텝 환경의 `task_description`을 읽어 정책에 넘기므로, 환경이 만들어질 때 이 값을 바꿉니다. 실제로 쓰인 지시문은 `instructions.json`에 기록됩니다.

In [ ]:
%%writefile /content/run_eval.py
"""lerobot-eval 래퍼: LIBERO 과제의 언어 지시문만 바꿔서 VLA를 평가한다.

사용법:
    LANG_MODE=<original|paraphrase|wrong_object|nonsense> LANG_LOG=<json 경로> \
        python run_eval.py [lerobot-eval 인자...]

장면, 초기 상태, 성공 판정은 원래 과제 그대로 두고 정책에 들어가는 문장만 바꾼다.
lerobot-eval은 매 스텝 env.task_description을 읽어 정책에 넘기므로, 환경이 생성될 때
이 값을 바꿔치기한다. 같은 프로세스에서 환경을 만들도록 --eval.use_async_envs=false로 실행한다.
"""

import json
import os
import re
from pathlib import Path

from lerobot.envs import libero as libero_env
from lerobot.scripts import lerobot_eval

MODE = os.environ.get("LANG_MODE", "original")
LOG_PATH = os.environ.get("LANG_LOG")
MODES = ("original", "paraphrase", "wrong_object", "nonsense")
NONSENSE = "what is the weather like today"
# LIBERO-Object 지시문 형식: "pick up the <물체> and place it in the <용기>"
PATTERN = re.compile(r"^pick (?:up )?the (.+?) and place it in the (.+)$", re.IGNORECASE)


def scene_objects(bddl_path):
    """BDDL 파일의 (:objects ...) 블록에서 물체 종류 이름을 순서대로 뽑는다."""
    block = re.search(r"\(:objects(.*?)\)", Path(bddl_path).read_text(), re.S).group(1)
    return [line.rsplit(" - ", 1)[1].strip() for line in block.splitlines() if " - " in line]


def make_instruction(original, bddl_path, mode):
    if mode == "original":
        return original
    if mode == "nonsense":
        return NONSENSE
    match = PATTERN.match(original.strip())
    if not match:
        return original  # 형식이 다른 과제는 바꾸지 않는다 (로그에서 확인 가능)
    target, container = match.groups()
    if mode == "paraphrase":
        return f"grab the {target} and put it into the {container}"
    if mode == "wrong_object":
        # 같은 장면에 실제로 놓여 있는 다른 물체를 지시한다
        skip = {target.replace(" ", "_"), container.replace(" ", "_")}
        others = [o for o in scene_objects(bddl_path) if o not in skip]
        if not others:
            return original
        return f"pick up the {others[0].replace('_', ' ')} and place it in the {container}"
    raise ValueError(f"LANG_MODE must be one of {MODES}, got {mode!r}")


_original_init = libero_env.LiberoEnv.__init__
_log = {}


def _patched_init(self, *args, **kwargs):
    _original_init(self, *args, **kwargs)
    used = make_instruction(self.task_description, self._task_bddl_file, MODE)
    _log[self.task_id] = {"task": self.task, "original": self.task_description, "used": used}
    self.task_description = used
    if LOG_PATH:
        Path(LOG_PATH).write_text(json.dumps(_log, indent=2, ensure_ascii=False))


libero_env.LiberoEnv.__init__ = _patched_init

if __name__ == "__main__":
    if MODE not in MODES:
        raise SystemExit(f"LANG_MODE must be one of {MODES}, got {MODE!r}")
    print(f"[run_eval] LANG_MODE={MODE}")
    lerobot_eval.main()

## 3. 실험 설정
- `N_EPISODES`: 과제당 반복 횟수. 10개 과제 × 4개 조건이므로 **5면 총 200 에피소드**입니다.
- `N_ACTION_STEPS`: 모델이 한 번 추론한 행동을 몇 스텝 실행할지. 체크포인트 기본값은 1(매 스텝 추론)이라 느려서 **10**으로 올렸습니다. 모든 조건에 똑같이 적용되므로 비교는 공정합니다. 다만 공개된 성공률과 숫자가 다를 수 있습니다.
- `RESULTS`: 결과 저장 위치. 런타임이 끊겨도 남도록 **Google Drive**에 저장합니다. 셀을 실행하면 Drive 연결 허용 창이 뜹니다.
- 같은 `SEED`를 쓰고 LIBERO의 고정 초기 상태를 사용하므로, 조건끼리 **같은 시작 장면**에서 비교됩니다.

In [ ]:
import json, os, subprocess, time
from pathlib import Path

os.environ["MUJOCO_GL"] = "egl"          # 화면 없는 서버에서 MuJoCo 렌더링
os.environ["PYOPENGL_PLATFORM"] = "egl"

POLICY = "HuggingFaceVLA/smolvla_libero"
SUITE = "libero_object"
CONDITIONS = ["original", "paraphrase", "wrong_object", "nonsense"]
N_EPISODES = 5
N_ACTION_STEPS = 10
SEED = 1000
# 런타임이 끊겨도 결과가 남도록 Google Drive에 저장한다
from google.colab import drive
drive.mount("/content/drive")
RESULTS = Path("/content/drive/MyDrive/vla_results_full")


def summarize(info):
    """eval_info.json에서 전체 성공 횟수, 성공률, 95% 윌슨 신뢰구간(%)을 계산한다."""
    successes = [s for task in info["per_task"] for s in task["metrics"]["successes"]]
    n, k, z = len(successes), sum(successes), 1.96
    p = k / n
    center = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * (p * (1 - p) / n + z * z / (4 * n * n)) ** 0.5 / (1 + z * z / n)
    return {"pc_success": 100 * p, "n_success": k, "n_episodes": n,
            "ci95": (100 * max(0, center - half), 100 * min(1, center + half)),
            "eval_ep_s": info["overall"]["eval_ep_s"]}


def run(mode, out_dir, n_episodes=N_EPISODES, task_ids=None):
    """한 조건을 평가한다. out_dir/eval_info.json이 이미 있으면 건너뛴다."""
    out_dir = Path(out_dir)
    if (out_dir / "eval_info.json").exists():
        print(f"[{mode}] 이미 완료됨 → 건너뜀")
        return
    out_dir.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        "/content/venv/bin/python", "/content/run_eval.py",
        f"--policy.path={POLICY}",
        f"--policy.n_action_steps={N_ACTION_STEPS}",
        "--env.type=libero",
        f"--env.task={SUITE}",
        "--eval.batch_size=1",
        f"--eval.n_episodes={n_episodes}",
        "--eval.use_async_envs=false",
        f"--seed={SEED}",
        f"--output_dir={out_dir}",
    ]
    if task_ids is not None:
        cmd.append(f"--env.task_ids={json.dumps(task_ids)}")
    # MPLBACKEND: Colab 커널의 그래프 설정(matplotlib_inline)이 가상환경으로 넘어가지 않게 막는다
    env = dict(os.environ, LANG_MODE=mode, LANG_LOG=str(out_dir.parent / f"{out_dir.name}_instructions.json"),
               MPLBACKEND="Agg")
    start = time.time()
    proc = subprocess.run(cmd, env=env, capture_output=True, text=True)
    (out_dir.parent / f"{out_dir.name}_log.txt").write_text(proc.stdout + proc.stderr)
    if proc.returncode != 0:
        print((proc.stdout + proc.stderr)[-3000:])
        raise RuntimeError(f"[{mode}] 실패 — 위 로그를 확인하세요")
    info = summarize(json.loads((out_dir / "eval_info.json").read_text()))
    print(f"[{mode}] 성공률 {info['pc_success']:.0f}% ({info['n_success']}/{info['n_episodes']}) "
          f"| 에피소드당 {info['eval_ep_s']:.0f}초 | 총 {(time.time() - start) / 60:.1f}분")
    return info

## 4. 짧은 동작 확인 (3~5분)
본 실험 전에 0번 과제를 **1번만** 돌려서 설치가 제대로 됐는지 확인하고, 본 실험에 걸릴 시간을 추정합니다.
처음에는 모델(약 2GB)을 내려받느라 더 오래 걸립니다.

In [ ]:
smoke = run("original", RESULTS / "smoke_test", n_episodes=1, task_ids=[0])
total_min = smoke["eval_ep_s"] * 10 * N_EPISODES * len(CONDITIONS) / 60
print(f"\n본 실험 예상 시간: 약 {total_min:.0f}분 (모델 다운로드 제외)")
if total_min > 120:
    print("→ 2시간이 넘습니다. 위 설정 셀에서 N_EPISODES를 3으로 줄이는 것을 권장합니다.")

In [ ]:
from IPython.display import Video
videos = sorted((RESULTS / "smoke_test" / "videos").rglob("*.mp4"))
Video(str(videos[0]), embed=True, width=480) if videos else print("영상이 없습니다")

## 5. 본 실험 실행
조건 4개를 차례로 돌립니다. 지난 실행 기준으로 **약 5.7시간**(original 63분, paraphrase 80분, wrong_object 98분, nonsense 약 100분)이 걸립니다. 실패한 에피소드는 제한 시간까지 다 돌기 때문에 4번의 예상 시간보다 길어집니다.

출력은 조건 하나가 끝날 때마다 한 줄씩 나옵니다. 진행 상황은 왼쪽 파일 패널의 `drive/MyDrive/vla_results_full/<조건>/videos/`에 영상이 늘어나는지로 확인하세요.

**중간에 끊기면** 1번(설치) → 2번 → 3번(설정) → 이 셀 순서로 다시 실행하세요. 끝난 조건은 Drive에 남아 있어 건너뛰고, 진행 중이던 조건만 처음부터 다시 돕니다.
**Colab 무료 버전은 브라우저 탭을 오래 방치하면 연결이 끊길 수 있으니** 가끔 탭을 확인해 주세요.

In [ ]:
for mode in CONDITIONS:
    run(mode, RESULTS / mode)

## 6. 결과 분석
### 조건별 · 과제별 성공률 (%)

In [ ]:
import pandas as pd

LABELS = {"original": "Original", "paraphrase": "Paraphrase",
          "wrong_object": "Wrong object", "nonsense": "Nonsense"}
rows, overall, instructions = [], {}, {}
for mode in CONDITIONS:
    path = RESULTS / mode / "eval_info.json"
    if not path.exists():
        print(f"[{mode}] 결과 없음 — 5번 셀을 먼저 실행하세요")
        continue
    info = json.loads(path.read_text())
    overall[mode] = summarize(info)
    instructions[mode] = json.loads((RESULTS / f"{mode}_instructions.json").read_text())
    for task in info["per_task"]:
        s = task["metrics"]["successes"]
        rows.append({"task_id": task["task_id"], "condition": LABELS[mode], "success": 100 * sum(s) / len(s)})

table = (pd.DataFrame(rows).pivot(index="task_id", columns="condition", values="success")
         .reindex(columns=[LABELS[m] for m in CONDITIONS if m in overall]))
table.insert(0, "original instruction", [instructions["original"][str(t)]["original"] for t in table.index])
table.loc["overall"] = [""] + [overall[m]["pc_success"] for m in CONDITIONS if m in overall]
styled = table.style.format(precision=0, na_rep="-")
styled = styled.background_gradient(cmap="Blues", vmin=0, vmax=100, subset=list(table.columns[1:]))
styled

### 조건별 전체 성공률 그래프
오차 막대는 95% 신뢰구간입니다. 에피소드 수가 적으면 구간이 넓으니, 막대끼리 겹치는 차이는 우연일 수 있습니다. 그림은 `/content/results/success_by_condition.png`로 저장되어 발표 자료에 바로 쓸 수 있습니다.

In [ ]:
import matplotlib.pyplot as plt

modes = [m for m in CONDITIONS if m in overall][::-1]   # 위에서부터 Original이 오도록
rates = [overall[m]["pc_success"] for m in modes]
lows = [overall[m]["pc_success"] - overall[m]["ci95"][0] for m in modes]
highs = [overall[m]["ci95"][1] - overall[m]["pc_success"] for m in modes]

fig, ax = plt.subplots(figsize=(7, 3.2), dpi=150)
ax.barh([LABELS[m] for m in modes], rates, height=0.55, color="#2a78d6",
        xerr=[lows, highs], error_kw={"ecolor": "#8a8a85", "elinewidth": 1, "capsize": 3})
for y, (m, r) in enumerate(zip(modes, rates)):
    ax.text(overall[m]["ci95"][1] + 2, y, f"{r:.0f}%  ({overall[m]['n_success']}/{overall[m]['n_episodes']})",
            va="center", fontsize=9, color="#3a3a38")
ax.set_xlim(0, 125)
ax.set_xticks(range(0, 101, 20))
ax.set_xlabel("Success rate on the original task (%)", color="#5f5e5a")
ax.set_title(f"SmolVLA on LIBERO-Object: same scene, different instruction", loc="left", fontsize=11)
ax.grid(axis="x", color="#e6e5e0", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color("#c3c2b7")
ax.tick_params(colors="#5f5e5a", length=0)
fig.tight_layout()
fig.savefig(RESULTS / "success_by_condition.png", bbox_inches="tight")
plt.show()

### 조건별 로봇 영상 비교
같은 과제, 같은 시작 장면에서 지시문만 바꿨을 때 로봇의 행동을 나란히 봅니다. `TASK_ID`와 `EPISODE`를 바꿔 다른 장면도 볼 수 있습니다.

In [ ]:
import base64
from IPython.display import HTML

TASK_ID, EPISODE = 0, 0
cards = []
for mode in CONDITIONS:
    clip = RESULTS / mode / "videos" / f"{SUITE}_{TASK_ID}" / f"eval_episode_{EPISODE}.mp4"
    if not clip.exists():
        continue
    ok = json.loads((RESULTS / mode / "eval_info.json").read_text())
    ok = next(t for t in ok["per_task"] if t["task_id"] == TASK_ID)["metrics"]["successes"][EPISODE]
    data = base64.b64encode(clip.read_bytes()).decode()
    cards.append(f'''<div style="width:300px;font-family:sans-serif;font-size:12px">
      <b>{LABELS[mode]}</b> — {"성공" if ok else "실패"}<br>
      <i>"{instructions[mode][str(TASK_ID)]["used"]}"</i><br>
      <video src="data:video/mp4;base64,{data}" width="300" controls loop muted></video></div>''')
HTML('<div style="display:flex;flex-wrap:wrap;gap:12px">' + "".join(cards) + "</div>")

## 7. 결과 해석 가이드 (발표용)

| 관찰 결과 | 해석 |
|---|---|
| `paraphrase` ≈ `original` | 표현이 바뀌어도 뜻을 유지 → 언어 이해에 어느 정도 일반화가 있음 |
| `paraphrase` ≪ `original` | 학습 때 본 **문장 형태 자체**에 의존 (표현에 취약) |
| `wrong_object`도 원래 과제 성공률이 높음 | 지시를 무시하고 **장면 배치만 보고 과제를 추측**함 |
| `nonsense`도 성공률이 높음 | 언어 입력이 사실상 쓰이지 않음 — 위와 같은 결론을 더 강하게 뒷받침 |
| `wrong_object`, `nonsense`에서 크게 떨어짐 | 정책이 실제로 언어를 근거로 행동을 고름 |

**왜 이런 일이 생길 수 있나?** LIBERO-Object는 과제마다 물체 배치가 달라서, 화면만 보고도 무슨 과제인지 알 수 있습니다. 그러면 모델은 굳이 문장을 읽지 않아도 학습 데이터를 잘 맞힐 수 있습니다. 벤치마크 점수가 높아도 "지시를 이해한다"는 보장은 없다는 뜻이고, 이것이 VLA의 일반화 한계와 WAM 같은 대안이 논의되는 배경과 연결됩니다.

**실험의 한계** (발표에서 함께 밝히면 좋습니다)
- 에피소드 수가 적어 신뢰구간이 넓습니다.
- `n_action_steps`를 10으로 바꿔서 공개 성능과 절대 수치가 다를 수 있습니다.
- `wrong_object`에서 로봇이 **지시한 물체를 실제로 옮겼는지**는 자동 판정하지 않습니다. 영상으로 직접 확인하세요.
- 모델 1개, 과제 묶음 1개에 대한 결과입니다.

## 8. 결과 내려받기
결과 폴더(영상, 그래프, 로그 포함)를 zip으로 묶어 내려받습니다.

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive("/content/vla_lang_test_results", "zip", RESULTS)
files.download(archive)